# Import libraries

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import re
import pandas as pd

# Cleaning functions

In [21]:
# Function to remove the first row (duplicated original column names)
def remove_first_row(df):
    return df.iloc[1:, :].copy()

# Usage:
# subset_1_information_samples = remove_first_row(subset_1_information_samples)





# Function to clean and standardize lab/sample IDs to FXX-XXXX format
def clean_lab_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'\s+', '-', regex=True)                 # Replace spaces with -
        .str.replace(r'^(F\d{2})(\d+)', r'\1-\2', regex=True) # Format FXX-XXXX
    )


# Usage:
# subset_1_information_samples['10_ciat_lab_id'] = clean_lab_ids(subset_1_information_samples['10_ciat_lab_id'])



# Function to clean and standardize Gene Bank - Breeding program IDs
def clean_standardize_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'-1$', '', regex=True)                   # Remove trailing -1
        .str.strip()                                           # Remove leading/trailing spaces
        .str.replace(r'[_\s]+', '-', regex=True)               # Replace spaces/underscores with -
        .str.replace(r'([A-Za-z])(\d)', r'\1-\2', regex=True)  # Letter followed by number
        .str.replace(r'(\d)([A-Za-z])', r'\1-\2', regex=True)  # Number followed by letter
        .str.replace(r'-+', '-', regex=True)                   # Remove repeated -
        .str.replace(r'^-|-$', '', regex=True)                 # Remove leading/trailing -
        .str.replace('ABC-', 'CIAT-', regex=False)             # Replace id's strings 'ABC' with 'CIAT'
    )

    # subset_1_information_samples['10_gene_bank_breeding_program_id'] = clean_standardize_ids(subset_1_information_samples['10_gene_bank_breeding_program_id'])


# 1. Shiny app training set
Requested by: Khaled Al-Sham'aa, (ICARDA)

Date: 2026_05_19

In [ ]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,3,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,62.577353,4.000526,11.156690,14.2,17.82863877,135.945555,227.8476704,37.051645,65.414752
1,1,4,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,61.249843,4.225907,11.205181,15,18.29422035,133.168047,270.0095702,38.185077,63.799940
2,1,5,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,29.500225,63.019856,4.071031,10.976075,13.8,17.41685196,137.016371,249.4607864,37.088353,64.343484
3,1,6,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,30.827735,67.002386,4.285055,11.302938,13.9,16.86945515,148.741403,175.9559634,41.714068,60.152044
4,1,7,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,29.500225,66.117380,4.130032,11.233760,14,16.99063023,146.659412,149.5197103,39.832832,62.557335


In [ ]:
# Filter by functional group 'Grass'
subsets_gas_grass = subsets_gas[subsets_gas['functional_group'] == 'Grass']
subsets_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,26.019199,58.536668,2.367747,6.302361,9.1,10.76651796,122.759406,194.6816788,27.337315,48.347519
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,27.789212,61.806682,2.445451,6.425495,8.8,10.39611646,129.591169,117.6219867,25.413695,53.012554
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,25.576695,62.094165,2.506516,6.742543,9.8,10.85857685,130.141925,195.7779286,28.971955,48.776690
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.169131,44.186601,1.837097,7.078486,10.7,16.01953085,92.531478,89.5309162,25.377242,58.411031
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,20.709158,51.726628,2.360844,8.502303,11.4,16.43699461,108.342750,110.6977836,30.734726,57.941924


In [ ]:
# Filter by subset '2'
subset_2_gas_grass = subsets_gas_grass[subsets_gas_grass['subset'] == 2]
subset_2_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
2373,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.945160,58.779199,3.246500,10.094461,15.5,18.1,NaN,NaN,NaN,NaN
2374,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.060153,55.239172,3.089264,9.667740,15.4,18.7,117.726263,#DIV/0!,164.369645,12.535150
2375,2,103,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.060153,57.451689,3.169504,10.161721,15.8,18.7,122.564018,#DIV/0!,60.146889,36.042450
2472,2,206,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,19.396398,47.915741,2.967649,7.901495,15.3,17.3,102.159310,#DIV/0!,47.734635,35.291931
2473,2,207,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,19.838901,50.039757,2.995674,8.099619,15.1,16.9,106.687844,#DIV/0!,52.077666,33.159869


In [ ]:
subset_2_gas_grass.batch.unique()

array([34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61])

In [ ]:
# Filter by batch =< 40
subset_2_gas_grass_batchs = subset_2_gas_grass[subset_2_gas_grass['batch'] >= 50]
subset_2_gas_grass_batchs.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
3870,2,1701,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,36.432778,79.886610,4.881992,12.225690,13.4,16.9,155.827612,34.19146976,43.344262,55.018936
3871,2,1702,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,36.875281,82.099127,5.125664,12.723270,13.9,16.8,155.999232,35.00485836,43.501476,55.574898
3872,2,1703,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,35.105268,79.444106,5.160474,12.653738,14.7,16.9,153.594861,35.25395301,43.886070,55.745158
3873,2,1704,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,35.105268,75.019073,5.090264,12.194921,14.5,17.8,148.711930,42.1835579,39.267646,61.562784
3874,2,1705,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,39.530302,82.984133,5.652833,13.170346,14.3,17.3,160.584546,36.28321101,44.216137,57.640147


In [ ]:
subset_2_gas_grass_batchs.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/ciat_gas_for_shiny_app_and_blues.csv', index = None)

# 2. Gas dashboard of Subset 4
Requested by: Alejandra Marín

Date: 2026_05_22

In [ ]:
dashboard_small = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_small.csv')
dashboard_small.head()

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,1,F24-3469,ABC-10647,Stylosanthes scabra,Herbaceous_legumes,15.90,18.79,48.31,67.34
1,1,F24-3470,ABC-11194,Stylosanthes hamata,Herbaceous_legumes,15.17,18.02,46.12,59.25
2,1,F24-3471,ABC-11999,Stylosanthes guianensis,Herbaceous_legumes,12.98,16.25,46.61,58.55
3,1,F24-3472,ABC-12318,Stylosanthes hamata,Herbaceous_legumes,13.99,16.96,52.27,56.88
4,1,F24-3427,ABC-1257,Stylosanthes scabra,Herbaceous_legumes,14.53,16.40,43.29,58.67


In [ ]:
# Filter data of subset 4
dashboard_small_subset_4 = dashboard_small[dashboard_small['subset'] == 4]

In [ ]:
# Order by id
def natural_sort_key(s):
    """Splits a string into alphanumeric components and converts numbers to integers for natural sorting."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', str(s))]

# Apply natural sort to the 'id' column
dashboard_small_subset_4_natural_sorted = dashboard_small_subset_4.sort_values(
    by='id',
    key=lambda col: col.apply(natural_sort_key)
).reset_index(drop=True)

display(dashboard_small_subset_4_natural_sorted.head())

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,4,F25-2640,Cayman-Br-02-1752,Urochloa interespecifico,Grass,12.99,13.71,50.17,50.22
1,4,F25-2574,CIAT-326,Desmodium scorpiurus,Herbaceous_legumes,13.86,16.66,55.45,39.09
2,4,F25-2575,CIAT-415,Vigna radiata,Herbaceous_legumes,11.22,15.38,43.25,51.39
3,4,F25-2576,CIAT-416,Vigna radiata,Herbaceous_legumes,12.38,16.84,50.26,56.45
4,4,F25-2577,CIAT-517,Macroptilium atropurpureum,Herbaceous_legumes,11.58,14.88,52.88,37.99


In [ ]:
#dashboard_small_subset_4_natural_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/subset_4_gas_dashboard_small.csv', index = None)

# 3. Grasses gas complete

Requested by: Claudia Perea
Date: 2026_05_26

In [ ]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,3,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,62.577353,4.000526,11.156690,14.2,17.82863877,135.945555,227.8476704,37.051645,65.414752
1,1,4,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,61.249843,4.225907,11.205181,15,18.29422035,133.168047,270.0095702,38.185077,63.799940
2,1,5,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,29.500225,63.019856,4.071031,10.976075,13.8,17.41685196,137.016371,249.4607864,37.088353,64.343484
3,1,6,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,30.827735,67.002386,4.285055,11.302938,13.9,16.86945515,148.741403,175.9559634,41.714068,60.152044
4,1,7,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,29.500225,66.117380,4.130032,11.233760,14,16.99063023,146.659412,149.5197103,39.832832,62.557335


In [ ]:
# Filter by functional group 'Grass'
grasses = subsets_gas[subsets_gas['functional_group'] == 'Grass']
grasses.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,26.019199,58.536668,2.367747,6.302361,9.1,10.76651796,122.759406,194.6816788,27.337315,48.347519
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,27.789212,61.806682,2.445451,6.425495,8.8,10.39611646,129.591169,117.6219867,25.413695,53.012554
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,25.576695,62.094165,2.506516,6.742543,9.8,10.85857685,130.141925,195.7779286,28.971955,48.776690
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.169131,44.186601,1.837097,7.078486,10.7,16.01953085,92.531478,89.5309162,25.377242,58.411031
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,20.709158,51.726628,2.360844,8.502303,11.4,16.43699461,108.342750,110.6977836,30.734726,57.941924


In [ ]:
grasses.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/grasses.csv', index = None)

# 4. Primary traits for metabolomics

Requested by : Jenny Gallo

Date: 2026_06_03


In [105]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/requested_jenny.csv')
data = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_complete.csv')

In [106]:
requested = remove_first_row(requested)
requested.head()

,approach,gene_bank_breeding_id,tax_name,functional_group,ch4_intensity_ml_g_tddm,tddm_percentage,ch4_intensity_decrease,ch4_percentage_8h,ch4_percentage_24h,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
1,Approach 1 (a),CIAT-772,Clitoria ternatea,Climber,37.9,65.1,41%,13.45555556,14.98567864,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Approach 1 (a),CIAT-19213,Clitoria ternatea,Climber,39.1,63.7,39%,14.98888889,16.08929282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Approach 2,CIAT-9434,Clitoria ternatea,Climber,46.51,51.78,33%,14.36666667,15.85424307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Approach 2,CIAT-955,Clitoria ternatea,Climber,45.45,59.65,30%,12.82222222,14.54174742,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Approach 4,CIAT-9432,Clitoria ternatea,Climber,57.24,50.87,11%,17.36,18.90480266,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
data.head()

,subset,requisitioner,no,id_lab,id,n_replicates,tax_order,family,genus,species,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,LMF,2202,F25-0769,AMC-ICARDA-156302,3,Fabales,Fabaceae,Trifolium,repens,...,86.99,6.03,14.08,15.97,16.37,3.75,30.99,-75.10,43.67,72.00
1,1,LMF,2235,F25-0770,AMC-ICARDA-165072,3,NaN,NaN,Vicia,tenuifolia,...,86.99,6.19,14.43,14.37,18.77,2.97,31.76,NaN,56.12,56.78
2,1,LMF,2226,F25-0773,AMC-ICARDA-168125,3,NaN,NaN,Lathyrus,sylvestris,...,61.77,4.70,10.30,15.53,17.77,3.12,22.66,NaN,55.10,42.32
3,1,Genetic bank,1057,F24-3469,CIAT-10647,6,Fabales,Fabaceae,Stylosanthes,scabra,...,79.20,6.90,14.88,15.90,22.28,3.95,32.01,151.04,48.31,67.34
4,1,Genetic bank,1060,F24-3470,CIAT-11194,6,Fabales,Fabaceae,Stylosanthes,hamata,...,70.41,5.14,12.69,15.17,20.65,3.91,27.31,184.66,46.12,59.25


In [110]:
requested.rename(columns={'gene_bank_breeding_id': 'id'}, inplace=True)
approach_id  = requested.iloc[:, :2]
approach_id.tail()

,approach,id
38,Approach 4,CIAT-9189
39,Approach 1 (a),CIAT-913
40,Approach 2,CIAT-18701
41,Approach 2,CIAT-20891
42,Approach 4,CIAT-18700


In [111]:
merged_df = approach_id.merge(data, on='id')
merged_df.tail()

,approach,id,subset,requisitioner,no,id_lab,n_replicates,tax_order,family,genus,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
34,Approach 4,CIAT-9189,2,Genetic bank,80,F24-3634,9,Fabales,Fabaceae,Indigofera,...,68.25,5.60,11.92,15.27,20.18,3.92,25.84,-6.88,45.56,57.89
35,Approach 1 (a),CIAT-913,1,Genetic bank,29,F24-3424,9,Fabales,Fabaceae,Cajanus,...,58.27,3.24,9.37,13.53,18.06,5.01,19.76,102.87,32.83,60.86
36,Approach 2,CIAT-18701,2,Genetic bank,963,F25-0934,9,Fabales,Fabaceae,Cajanus,...,47.08,3.14,7.11,13.36,16.87,3.18,14.99,-14.45,54.17,31.19
37,Approach 2,CIAT-20891,2,Genetic bank,966,F25-0935,9,Fabales,Fabaceae,Cajanus,...,45.26,3.38,6.54,12.92,16.56,3.11,13.72,-30.93,47.98,29.51
38,Approach 4,CIAT-18700,2,Genetic bank,1322,F25-0978,9,Fabales,Fabaceae,Cajanus,...,53.11,3.87,8.24,14.00,17.27,2.84,17.47,-25.22,59.27,32.23


In [112]:
merged_df.columns

Index(['approach', 'id', 'subset', 'requisitioner', 'no', 'id_lab',
       'n_replicates', 'tax_order', 'family', 'genus', 'species', 'tax_name',
       'functional_group', 'set_ciat', 'batch', 'run', 'replication',
       'syrange', 'sample_weight_g', 'undigested_dm_g', 'dm_incubated',
       'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml', 'ch4_8h_ml',
       'ch4_24h_ml', 'ch4_8h_percentage', 'ch4_24h_percentage', 'part_fact',
       'ch4_ml_g_dm_incubaed_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm'],
      dtype='object')

In [114]:
primary_traits = merged_df[['approach', 'id','tax_name','functional_group', 'dm_incubated',  'digested_feed_mg', 'ch4_24h_ml']]
primary_traits.head(50)

,approach,id,tax_name,functional_group,dm_incubated,digested_feed_mg,ch4_24h_ml
0,Approach 1 (a),CIAT-772,Clitoria ternatea,Herbaceous_legumes,457.74,298.00,11.18
1,Approach 1 (a),CIAT-19213,Clitoria ternatea,Herbaceous_legumes,469.33,298.75,11.57
2,Approach 2,CIAT-9434,Clitoria ternatea,Herbaceous_legumes,454.10,236.60,13.39
3,Approach 2,CIAT-955,Clitoria ternatea,Herbaceous_legumes,459.34,274.00,12.25
4,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,456.56,252.63,13.41
5,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,477.33,229.65,13.71
6,Approach 4,CIAT-18447,Clitoria ternatea,Herbaceous_legumes,464.97,257.04,13.64
7,Approach 1 (a),CIAT-7317,Canavalia sp.,Herbaceous_legumes,462.50,321.35,10.57
8,Approach 1 (a),CIAT-8719,Canavalia sp.,Herbaceous_legumes,467.83,326.25,10.94
9,Approach 2,CIAT-20803,Canavalia sp.,Herbaceous_legumes,467.17,260.21,10.92


In [116]:
#primary_traits.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/primary_traits_to_iomicas.csv', index=None)